# Reproducing Brudner, Zhou et al. (2026) from dandiset 001871

**Paper.** Brudner S, Zhou B, Jayaram V, Madeira Santana G, Crimaldi J, Clark DA, Emonet T (2026).
Fly navigational responses exploit plume-specific odor motion and gradient cues. *PNAS* 123(29)
e2608896123. [doi:10.1073/pnas.2608896123](https://doi.org/10.1073/pnas.2608896123). The claims file
(`claims/brudner-2026.yaml`) was written from the version of record and its SI Appendix; this attempt
read the same text, under `paper/`.

**Dandiset.** [DANDI:001871](https://dandiarchive.org/dandiset/001871/0.260630.1657), version
0.260630.1657. The paper cites no version. This is the only published version and the one the paper's
code (github.com/emonetlab/GradientMotionMultiplePlumes, commit effc7b0, `metadata/published_panel_params.json`)
names as the source of its Fig. 5 tables. The draft has the same 136 assets and total size.

**What the dandiset holds.** 131 behavior NWB files, one per trial of a fly cohort (83 smooth-plume
files, "nagel-smoke"; 24 "smoke-1a" and 24 "smoke-2a" complex-plume files), with per-trajectory
position, heading, velocities and the odor intensity, gradient and motion signals at the antennae,
and a table of turn events; the complex plume movie window (frames 300 to 1799 of smoke_1a_orig);
three freestream plume windows; and a compact asset of Fig. S1 panel inputs. The smooth plume movie
is not in it: it is on Dryad (doi:10.5061/dryad.g27mq71), which answers 401 to downloads without an
API token, and it was not staged for this run, so every claim that needs it is blocked. The paper's
Yale Dataverse deposit (doi:10.60600/YU/590UBV) holds the behavior tables only.

**Outside inputs.** None are used for the results. Method details come from the paper's code at the
commit above (cited in `plumes.py`).

| Claim | Tested here |
|---|---|
| fig5e-cue-betas | tested |
| fig5f-cue-dominance | tested |
| figs5c-turn-angles | tested |
| fig2c-complex-auc | tested |
| fig2e-complex-weights | tested |
| figs1fg-resolution-robustness | tested |
| methods-intensity-right-bias | blocked, metadata_insufficient (the direction code has no left/right meaning); numbers shown under fig5e |
| figs2c-one-frame-delay | blocked (smooth plume not readable); the complex part is shown |
| fig1eh, fig1il, figs1-antisymmetry-index, figs1-temporal-statistics, fig2b, fig2d | blocked (smooth plume not readable) |
| fig3bc, fig3d, fig3e, fig3ij, fig4ef, discussion-ten-percent-gain, figs4, figs5fh | blocked (smooth plume not readable; networks and agent simulations depend on it) |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd

import plumes

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
assets = plumes.list_assets()
assets["plume"] = assets.path.map(plumes.plume_of)
behavior_assets = assets[assets.plume.notna()]
sessions = plumes.load_behavior(behavior_assets, workers=8)
print(f"dandiset {plumes.DANDISET_ID} version {plumes.DANDISET_VERSION}: {len(assets)} assets; "
      f"loaded {len(sessions)} behavior files in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
kind = np.where(assets.plume.notna(), "behavior " + assets.plume.fillna(""), assets.path.str.split("/").str[0])
print("Assets by kind (count, GB):")
print(assets.assign(kind=kind).groupby("kind")["size"].agg(["count", "sum"]).assign(sum=lambda d: (d["sum"] / 1e9).round(2)).to_string())

info = pd.DataFrame([s["info"] for s in sessions])
info["trajectories"] = [s["frames"].trj.nunique() for s in sessions]
info["frames"] = [len(s["frames"]) for s in sessions]
info["turns"] = [len(s["turns"]) for s in sessions]
info["date"] = info.start.str[:10]
print("\nBehavior files by plume group:")
print(info.groupby("plume")[["trajectories", "frames", "turns"]].agg(["count", "sum"]).to_string())
print("\nFiles by recording date and group:")
print(pd.crosstab(info.date, info.plume).to_string())
print("\nDistinct session ids:", info.session_id.nunique(), "; distinct subject ids:", info.subject_id.nunique())
print("Series timing (rate or timestamps):", info.rates.astype(str).value_counts().to_dict())
print("Units stated:", sessions[0]["info"]["units"], "; position:", sessions[0]["info"]["xy_unit"])
print("Example session descriptions:", info.description.str.replace("Olfactory navigation behavior session: ", "").head(4).tolist())

The paper reports no count of cohorts, flies, trajectories or turns. Each file is one 2-minute trial
(session descriptions name the vial, the flip of the projection and a trial index); trajectories are
fly tracks, not flies, and a fly cannot be followed across trajectories, so the unit of clustering is
the trajectory, as in the paper (SI, Behavioral Analysis). The two complex groups were recorded on the
same days and vials (smoke_1a and smoke_2a), four files per vial and day for each.

In [ ]:
frames_all = pd.concat([s["frames"].assign(plume=s["info"]["plume"]) for s in sessions], ignore_index=True)
cols = ["x", "y", "theta", "dtheta", "vx", "spd", "signal", "spatial_gradient", "odor_velocity"]
print("Ranges of every variable used, by group (1st, 50th, 99th percentile, min, max, NaN fraction):")
rng_tab = frames_all.groupby("plume")[cols].agg([lambda v: np.nanpercentile(v, 1), "median", lambda v: np.nanpercentile(v, 99), "min", "max", lambda v: v.isna().mean()])
rng_tab.columns = [f"{c}|{s}" for c, s in zip(rng_tab.columns.get_level_values(0), ["p1", "p50", "p99", "min", "max", "nan"] * len(cols))]
print(rng_tab.T.round(3).to_string())
print("\nArena (Methods): 270 x 170 mm. Frames with y > 190 mm:", int((frames_all.y > 190).sum()),
      "; their largest y:", round(frames_all.y.max(), 1), "mm, in group(s)", frames_all.loc[frames_all.y > 190, "plume"].unique().tolist())
print("  files:", frames_all.loc[frames_all.y > 190].groupby("session").size().to_dict())

# all-NaN trajectories and repeated recordings
traj = frames_all.groupby(["plume", "session", "trj"]).agg(n=("x", "size"), nan=("x", lambda v: v.isna().mean()))
print("\nTrajectories with no position at all:", int((traj.nan == 1).sum()), "of", len(traj))
empty = set(traj.index[traj.nan == 1].droplevel(0))
n_on_empty = sum(int(s["turns"].apply(lambda r: (s["info"]["session_id"], r.trj) in empty, axis=1).sum()) for s in sessions)
print("Turn events on those trajectories:", n_on_empty)
import hashlib
hashes = {}
for (pl, se, tr), g in frames_all.groupby(["plume", "session", "trj"]):
    if g.x.notna().any():
        hashes.setdefault(hashlib.md5(np.round(g[["x", "y", "theta"]].to_numpy(), 4).tobytes()).hexdigest(), []).append((se, tr))
dups = [v for v in hashes.values() if len(v) > 1]
print("Trajectories with position whose x, y and heading repeat another trajectory exactly:", len(dups))
th = {}
for s in sessions:
    th.setdefault(hashlib.md5(np.round(s["turns"][["start_time", "turn_x", "turn_y"]].to_numpy(), 4).tobytes()).hexdigest(), []).append(s["info"]["session_id"])
print("Turn tables repeated across files:", [v for v in th.values() if len(v) > 1])
ids = frames_all[["session", "trj"]].drop_duplicates()
print("Trajectory ids unique across files:", ids.trj.is_unique, "(clusters are named by file and id)")

No recording is deposited twice: no trajectory with position repeats another, and no turn table
repeats. 205 trajectories have no position at all (they are left out of the comparison of repeats, as
any two all-NaN trajectories of equal length look identical); the number of turn events on them is
printed above. Four smooth files have 10 frames with positions far outside the 170 mm
arena width (y up to 814 mm), an error of the tracking or the conversion; these frames fall outside the analysis region of
Fig. 5 (y 67 to 97 mm) and do not enter any test. `odor_velocity` (motion) is defined on only about 3%
of frames; the code's exponential filter holds the last value across the gaps.

In [ ]:
moving = frames_all[(frames_all.spd > 5) & frames_all.theta.notna()]
print("Heading against the direction of travel (frames with speed > 5 mm/s):")
for name, ang in [("atan2(vy, vx)", np.arctan2(moving.vy, moving.vx)), ("atan2(-vy, vx)", np.arctan2(-moving.vy, moving.vx))]:
    diff = (moving.theta - np.degrees(ang) % 360 + 180) % 360 - 180
    print(f"  theta against {name}: median |difference| {np.median(np.abs(diff)):.1f} deg; {(np.abs(diff) < 30).mean():.3f} of frames within 30 deg")
print("Upwind is 180 deg (code: facing upwind 160 to 200; walking upwind vx < 0). Median vx of frames heading 160-200 deg at speed > 5 mm/s:",
      round(float(moving.vx[(moving.theta > 160) & (moving.theta < 200)].median()), 2), "mm/s")

opts0 = plumes.TurnOptions()
check = []
for s in sessions[:: 6]:
    p = plumes.prepare_frames(s["frames"], opts0)
    tt = plumes.turn_table(p, s["turns"], onset="start")
    check.append(tt.assign(plume=s["info"]["plume"]))
check = pd.concat(check)
print(f"\nTurn events, from every 6th file ({len(check)} turns):")
print("sign of dtheta at the peak of |dtheta| within the event, against turn_direction:")
print(pd.crosstab(np.sign(check.dtheta_peak), check.direction).to_string())
print("event duration (frames):", ((check.stop - check.start) * 60).round().value_counts().sort_index().to_dict())
lead = ((check.start - check.onset20) * 60).round()
print("frames from the SI onset (20% of the peak on the rising edge) to the event start:", lead.describe().round(2).to_dict())
print("turn_theta equals theta_smooth at the event start; turn_x, turn_y equal x, y there (checked on 300 turns in preparation, median |difference| 0)")

The heading is the direction of travel measured with y reversed, theta = atan2(-vy, vx) (not
atan2(vy, vx)), and 180 degrees is travel towards -x, upwind, as the code assumes; the files say only
"standardized arena coordinates". The turn direction (+1 or -1) is the sign of the rotational velocity
at the event's peak, so +1 turns are counterclockwise in the (x, -y) frame. Whether that is a left or
a right turn depends on how the camera viewed the arena, which the files do not state, and the
direction column is described only as "Turn direction label"; `methods-intensity-right-bias` needs
that and is blocked. The event start is not
the SI's onset: it comes a median of 4 frames (67 ms) after the time rotational speed crosses 20% of
the peak (from every 6th file above). The claims file asks for the SI's onset in that case, so it is
the main reading below, and the code's use of the event start is the other.

**Clock.** The turn events share the trajectories' clock: every event's peak of |dtheta| has the
sign of its direction. The odor signals were computed by the authors from the projected movie, and
the files give no movie frame for a trajectory frame, so whether they share the movie's clock cannot
be shown here; a scan of lags between the files' `signal` and the deposited complex window, with the
trial time taken modulo the 60 s movie, found no consistent peak (correlations about 0.1 to 0.2,
best lags at the edges of +/-30 frames, in preparation, not run here).

In [ ]:
# Fig. 5 turn tables: every plume group, both onsets. Frames are prepared once per group.
t0 = time.time()
groups = {"smooth": ["smooth"], "complex_2a": ["complex_2a"], "complex_1a": ["complex_1a"]}
prepared, turns = {}, {}
for g in groups:
    sub = [s for s in sessions if s["info"]["plume"] == g]
    prepared[g] = plumes.prepare_frames(pd.concat([s["frames"] for s in sub], ignore_index=True), opts0)
    turns[g] = pd.concat([s["turns"] for s in sub], ignore_index=True)
prepared["complex_both"] = pd.concat([prepared["complex_1a"], prepared["complex_2a"]], ignore_index=True)
turns["complex_both"] = pd.concat([turns["complex_1a"], turns["complex_2a"]], ignore_index=True)
del frames_all, moving
for s in sessions:
    s["frames"] = None    # free memory: everything below uses the prepared frames
TT = {(g, on): plumes.turn_table(prepared[g], turns[g], onset=on) for g in prepared for on in ("rule20", "start")}
print(f"turn tables built in {time.time() - t0:.0f} s")
for (g, on), tt in TT.items():
    print(f"{g:13s} onset={on:6s} turns {len(tt):6d}, matched to a frame {tt.matched.sum():6d}")

## fig5e-cue-betas

> For upwind-facing flies in the bulk of the plume, the gradient cue 50 ms before a turn significantly
> predicts turn direction in the smooth plume while motion does not, and the motion cue does in the
> complex plume while the gradient does not.

Readings (claims file): heading window in the complex plume (160 to 200, SI; 150 to 210, code); the
code's 15 mm margin (not applied; applied); z-scoring (none, code; pooled over the plume's frames;
per trajectory); "not significant" (p >= 0.05; p >= 0.01). Onset: the SI's 20% rule (main, as the
claims file asks when the event start differs) and the event start (code). Which complex files the
paper used is not stated: the code takes "smoke-2a", the main reading here, as the code produced the
figure; "smoke-1a" and both are shown. Expected values are compared under the code's readings
(event start, margin, no z-scoring, complex heading 150 to 210, smoke-2a).

In [ ]:
rows = []
fits5 = {}
for (g, on), tt in TT.items():
    headings = [(160.0, 200.0)] if g == "smooth" else [(160.0, 200.0), (150.0, 210.0)]
    for hd in headings:
        for margin in (False, True):
            for z in ("none", "pooled", "per_trajectory"):
                o = plumes.TurnOptions(heading=hd, margin=margin, zscore=z)
                k = plumes.keep(tt, o)
                r = plumes.fig5_models(k)
                f = r["fits"]["full"]
                fits5[(g, on, hd, margin, z)] = r
                rows.append({"group": g, "onset": on, "heading": f"{hd[0]:.0f}-{hd[1]:.0f}", "margin": margin, "z": z,
                             "turns": int(f.nobs), "trajectories": k.cluster.nunique(),
                             "b_grad": f.params["gradient"], "se_grad": f.bse["gradient"], "p_grad": f.pvalues["gradient"],
                             "b_motion": f.params["motion"], "se_motion": f.bse["motion"], "p_motion": f.pvalues["motion"],
                             "b_int": f.params["intensity"], "p_int": f.pvalues["intensity"],
                             "dominance": r["dominance"], **{f"ll_{m}": v for m, v in r["ll"].items()}})
R5 = pd.DataFrame(rows)
print(R5.drop(columns=[c for c in R5 if c.startswith("ll_")]).round(4).to_string(index=False))

In [ ]:
def smooth_pass(r, alpha):
    return (r.p_grad < 0.005) and (r.p_motion >= alpha)
def complex_pass(r, alpha):
    return (r.p_motion < 0.01) and (r.p_grad >= alpha)

summ = []
for g in ["smooth", "complex_2a", "complex_1a", "complex_both"]:
    sub = R5[R5.group == g]
    for alpha in (0.05, 0.01):
        ok = sub.apply(lambda r: smooth_pass(r, alpha) if g == "smooth" else complex_pass(r, alpha), axis=1)
        summ.append({"group": g, "not significant at": f"p >= {alpha}", "readings": len(sub), "pass": int(ok.sum())})
print("Readings in which each plume's part of the criterion holds:")
print(pd.DataFrame(summ).to_string(index=False))

print("\nComplex plume, motion coefficient over all readings: range", R5[R5.group != "smooth"].groupby("group").b_motion.agg(["min", "max"]).round(4).to_dict("index"))
print("Complex plume, smallest p of the motion coefficient per group:", R5[R5.group != "smooth"].groupby("group").p_motion.min().round(4).to_dict())
print("Smooth plume, largest p of the gradient coefficient:", round(R5[R5.group == "smooth"].p_grad.max(), 5),
      "; smallest p of motion:", round(R5[R5.group == "smooth"].p_motion.min(), 4))

code_rd = lambda g, hd: R5[(R5.group == g) & (R5.onset == "start") & R5.margin & (R5.z == "none") & (R5.heading == hd)].iloc[0]
cs, cc = code_rd("smooth", "160-200"), code_rd("complex_2a", "150-210")
exp = [("gradient, smooth", cs.b_grad, -0.1181, 0.0331), ("motion, smooth", cs.b_motion, -0.0074, 0.0114),
       ("motion, complex", cc.b_motion, 0.0441, 0.0159), ("gradient, complex", cc.b_grad, 0.0185, 0.0161)]
print("\nUnder the code's readings (event start, margin, no z-scoring, complex smoke-2a at 150-210):")
for name, got, want, tol in exp:
    print(f"  {name:18s} measured {got:+.4f}  expected {want:+.4f} +/- {tol:.4f}  within: {abs(got - want) <= tol}")
print(f"  turns kept: smooth {cs.turns}, complex {cc.turns}; p: smooth gradient {cs.p_grad:.5f}, smooth motion {cs.p_motion:.3f}, "
      f"complex motion {cc.p_motion:.3f}, complex gradient {cc.p_grad:.3f}")

**Result: reproduced, resting on assumptions.** The readings disagree, so the outcome is that of the
best-supported one: the SI's onset (the claims file's main reading, since the files' turn start is a
median of 4 frames later), the code's complex files (smoke-2a), and the predictors as stored or z-scored over all
frames (which gives the same p-values). Under it, in the smooth plume the gradient coefficient has p
below 0.0001 and motion p 0.22; in the complex plume the motion coefficient has p 0.0051 (heading 160
to 200) or 0.0057 (150 to 210) and the gradient p 0.14 or 0.26, with or without the margin. The pattern
fails with the files' turn start (complex motion p 0.12 to 0.95), with per-trajectory z-scoring, and
when the smoke-1a files are added to the smoke-2a ones (no reading passes); the smoke-1a files alone
pass in 10 of 24 readings, with a positive motion coefficient. The complex motion coefficient is negative here (-0.024 to -0.028
as stored), opposite in sign to the published +0.044; the criterion tests significance only. Under the
code's own readings the smooth coefficients match the expected values and the complex ones do not
(motion -0.012 against +0.044 +/- 0.016), so the code's published complex table does not come from these
files through this code.

## fig5f-cue-dominance

> Cue dominance, the gradient-minus-motion log-likelihood difference relative to the total bilateral
> contribution, is near 1 in the smooth plume (gradient dominates) and negative in the complex plume
> (motion dominates).

The four models and the measure are the code's (`VARIANT_FORMULAS`); the readings are those of
fig5e-cue-betas, paired so that both plumes use the same onset, margin and z-scoring.

In [ ]:
ll_cols = ["ll_full", "ll_gradient", "ll_motion", "ll_base"]
print("Mean log-likelihood per turn and dominance, every reading:")
print(R5[["group", "onset", "heading", "margin", "z", "turns"] + ll_cols + ["dominance"]].round(4).to_string(index=False))

pairs = []
for g in ["complex_2a", "complex_1a", "complex_both"]:
    for _, c in R5[R5.group == g].iterrows():
        s = R5[(R5.group == "smooth") & (R5.onset == c.onset) & (R5.margin == c.margin) & (R5.z == c.z)].iloc[0]
        pairs.append({"complex files": g, "onset": c.onset, "complex heading": c.heading, "margin": c.margin, "z": c.z,
                      "smooth": s.dominance, "complex": c.dominance,
                      "signs hold": (s.dominance > 0) and (c.dominance < 0),
                      "both within tolerance": abs(s.dominance - 0.946) <= 0.095 and abs(c.dominance + 0.662) <= 0.066})
PR = pd.DataFrame(pairs)
print("\nPaired readings (expected: smooth 0.946 +/- 0.095, complex -0.662 +/- 0.066):")
print(PR.round(3).to_string(index=False))
print("\nReadings where the signs hold / both values within tolerance, by complex file set:")
print(PR.groupby("complex files")[["signs hold", "both within tolerance"]].agg(["sum", "count"]).to_string())
print("\nUnder the code's readings: smooth", round(cs.dominance, 3), ", complex (smoke-2a)", round(cc.dominance, 3))
print("Smooth dominance over all readings: range", round(R5[R5.group == 'smooth'].dominance.min(), 3), "to", round(R5[R5.group == 'smooth'].dominance.max(), 3))

**Result: reproduced, resting on assumptions.** As for fig5e-cue-betas, the readings disagree and the
best-supported one decides: with the SI's onset and the code's smoke-2a files, the smooth dominance is
0.935 and the complex dominance -0.715 with the code's complex heading window (150 to 210), both within
tolerance, or -0.579 with the SI's (160 to 200), negative but just outside it (z-scoring as stored or
over all frames, either margin). With the files' turn start, which the code uses, the complex
dominance is positive (0.39 under the code's readings, against -0.662), and per-trajectory z-scoring
changes both plumes. The log-likelihood differences behind the complex value are small (the full and
base models differ by 0.0055 to 0.0084 per turn).

## figs5c-turn-angles

> The detected saccade-like turns reorient flies by about 30 degrees, with median angular
> displacements of -34.2 and +33.8 degrees for the two turn directions.

Split by the sign of dtheta at the peak (Fig. S5C legend). Readings (claims file): all detected turns
of both plumes pooled, or the turns kept in fig5e; displacement as the heading change from the event's
start to its end, or the integral of rotational velocity over the event. The files' events start a median of
4 frames after the SI's turn onset (Data check), so the event as the paper bounds it is not the files' event.
The integral is therefore also run over the turn as the SI bounds its onset, from the 20% crossing on
the rising edge to the 20% crossing on the falling edge (the falling edge is not stated; taken as the
mirror of the onset rule, an assumption).

In [ ]:
def medians(tt):
    disp = (tt.theta_stop - tt.theta_start + 180) % 360 - 180
    out = {}
    for name, v in [("heading change", disp), ("integral of dtheta", tt.int_dtheta), ("integral over the SI-bounded turn", tt.int_dtheta_20)]:
        out[name] = (round(float(v[tt.dtheta_peak < 0].median()), 1), round(float(v[tt.dtheta_peak > 0].median()), 1))
    return out

main_s = plumes.TurnOptions()
sets = {
    "all turns, smooth + complex smoke-2a": pd.concat([TT[("smooth", "start")], TT[("complex_2a", "start")]]),
    "all turns, smooth + both complex": pd.concat([TT[("smooth", "start")], TT[("complex_both", "start")]]),
    "kept turns (fig5e main: SI onset, 160-200, no margin), smooth + smoke-2a": pd.concat([plumes.keep(TT[("smooth", "rule20")], main_s), plumes.keep(TT[("complex_2a", "rule20")], main_s)]),
}
rows = []
for name, tt in sets.items():
    for meas, (neg, pos) in medians(tt).items():
        rows.append({"turns": name, "n": len(tt), "measure": meas, "median, negative": neg, "median, positive": pos,
                     "within tolerance": abs(neg + 34.2) <= 3.42 and abs(pos - 33.8) <= 3.38})
print(pd.DataFrame(rows).to_string(index=False))
for g in ["smooth", "complex_2a", "complex_1a"]:
    print(g, "by group, all turns:", medians(TT[(g, "start")]))
allt = pd.concat([TT[("smooth", "start")], TT[("complex_both", "start")]])
d = (allt.theta_stop - allt.theta_start + 180) % 360 - 180
print("Range of the heading change over all turns:", round(d.min(), 1), "to", round(d.max(), 1), "deg; 5th and 95th percentiles of |change|:",
      np.percentile(np.abs(d), [5, 95]).round(1))

**Result: reproduced, resting on an assumption.** Over the files' events the medians are -20 to -21
and +21 to +22 degrees under both displacements and both turn sets, outside the tolerance of -34.2 +/-
3.4 and +33.8 +/- 3.4. Integrated over the turn as the SI bounds it (the 20% crossings of the peak),
they are -35.0 and +35.9 degrees for all turns pooled and -33.1 and +36.6 for the kept turns, within
tolerance. The outcome rests on the turn's end being the 20% crossing on the falling edge, which the
paper does not state.

## The complex plume movie

One streaming pass gives the mean intensity map and the time-averaged cue maps of Fig. S1F-G; a
second samples the antennae at the Fig. 2 locations, which are drawn from the mean map. See
`plumes.movie_pass1` and `plumes.sample_antennas` for the definitions.

In [ ]:
t0 = time.time()
movie, movie_file = plumes.open_movie(assets, *plumes.COMPLEX_MOVIE)
print("movie", movie.shape, movie.dtype, "chunks", movie.chunks)
P1 = plumes.movie_pass1(movie)
mean_map = P1["mean"]
print(f"pass 1: {P1['n_frames']} frames in {time.time() - t0:.0f} s")
print("intensity: mean map range", round(mean_map.min(), 2), "to", round(mean_map.max(), 2),
      "; values over all frames:", P1["frame_min"], "to", P1["frame_max"])
print("SI: 1500 frames of 1088 x 1728 pixels, 0 to 255; centerline row", plumes.COMPLEX_CENTERLINE)
print("row of the maximum of the mean map over the first 100 columns (source end):", int(np.argmax(mean_map[:, :100].mean(1))))
from skimage.filters import threshold_otsu
print("Otsu threshold of the mean map:", round(float(threshold_otsu(mean_map)), 2))
masks = {"mean > 5": mean_map > 5,
         "Otsu, bias 48 subtracted": plumes.otsu_mask(mean_map, 48, "subtract"),
         "Otsu, bias 48 %": plumes.otsu_mask(mean_map, 48, "percent")}
for k, m in masks.items():
    print(f"  valid region '{k}': {m.mean():.3f} of pixels")

The stored window matches the SI and the dandiset's description (1500 frames, 1088 x 1728, 0 to 255).
Its provenance names a preprocessing run "bgsub_gauss3_fullslice_p001_p99", so it may differ from the
movie the paper analysed (the SI describes a linear rescaling from 0-120 to 0-255 only).

## figs1fg-resolution-robustness

> The spatial organization of the gradient and motion cues in the complex plume stays consistent when
> sensor spacing is increased from 3 to 12 pixels or the motion delay is lengthened to about 150 ms,
> changing only when sampled more coarsely than the smooth plume.

Gradient of spacing d: the time average of B - A for sensors d rows apart. The motion cue for a
spacing and a delay is not defined in the paper; two readings are run (`plumes.movie_pass1`): the
Fig. 1 cue -(dI/dy)(dI/dt) with the spacing and the delay widened (main, as Fig. S1G is the motion cue
of column iv, `<-dI/dy dI/dt>_t`), and the two-sensor correlator A(t - tau)B(t) - A(t)B(t - tau).
Pearson r against the 3-pixel map (gradient) and the 3-pixel, 1-frame map (motion), over every pixel
of every 4th column, and as a variant over the Otsu mask. A further variant smooths the mean map by
the antennae's Gaussian (SD 1.5 pixels) before the gradient (the gradient's time average is the
difference of the mean map, so this needs no second pass).

In [ ]:
from scipy.ndimage import gaussian_filter
def r_(a, b, m=None):
    a, b = a.ravel(), b.ravel()
    if m is not None:
        a, b = a[m.ravel()], b[m.ravel()]
    return float(np.corrcoef(a, b)[0, 1])

mask_sub = masks["Otsu, bias 48 subtracted"][:, ::P1["col_step"]][6:-7]
grad_rows = []
smoothed = gaussian_filter(mean_map, 1.5)[:, ::P1["col_step"]]
gs = {d: (lambda a, b: b - a)(*plumes._pair(smoothed, d)) for d in plumes.SPACINGS}
for d in plumes.SPACINGS:
    grad_rows.append({"spacing px": d, "r": r_(P1["grad"][3], P1["grad"][d]), "r in mask": r_(P1["grad"][3], P1["grad"][d], mask_sub),
                      "r, smoothed sigma 1.5 (variant)": r_(gs[3], gs[d])})
G = pd.DataFrame(grad_rows)
print("Gradient maps against the 3-pixel map:")
print(G.round(3).to_string(index=False))
mot = {}
for form in ("grad_dt", "corr"):
    tab = pd.DataFrame({tau: {d: r_(P1["motion"][(form, 3, 1)], P1["motion"][(form, d, tau)]) for d in plumes.SPACINGS} for tau in plumes.DELAYS})
    tabm = pd.DataFrame({tau: {d: r_(P1["motion"][(form, 3, 1)], P1["motion"][(form, d, tau)], mask_sub) for d in plumes.SPACINGS} for tau in plumes.DELAYS})
    mot[form] = tab
    print(f"\nMotion maps ({form}) against the 3-pixel, 1-frame map; rows spacing (px), columns delay (frames):")
    print(tab.round(3).to_string())
    print("  in the Otsu mask:")
    print(tabm.round(3).to_string())
for form, tab in mot.items():
    ok_g = bool((G.r >= 0.8).all())
    ok_m = bool((tab[[1, 2, 4]] >= 0.8).all().all())
    ok_drop = bool(((tab[8] < tab[4]) & (tab[15] < tab[4])).all())
    print(f"\n{form}: gradient r >= 0.8 at every spacing: {ok_g}; motion r >= 0.8 at every spacing for delays 1, 2, 4: {ok_m}; "
          f"r at 8 and 15 frames below the 4-frame value at every spacing: {ok_drop}")
print("Figure values (claims file): gradient r about 0.89 at 12 px; motion r near 1 at 1, 2, 4 frames, about 0.65 at 8 and 0.35 at 15.")

**Result: not reproduced.** The gradient map's correlation with the 3-pixel map is 0.82, 0.70 and 0.61
at 6, 9 and 12 pixels (the figure shows about 0.89 at 12), below the criterion's 0.8 at 9 and 12 pixels,
in the whole frame and in the mask; with the mean map smoothed by the antennae's Gaussian (variant) it is
0.97, 0.91 and 0.81. The motion maps fall to 0.72 to 0.82 at 4 frames under both definitions of the cue,
where the figure shows r near 1, so the motion part fails under every reading; the drop at 8 and 15
frames below the 4-frame value holds.

## fig2c-complex-auc and fig2e-complex-weights

> In the complex plume, a logistic model on the odor motion feature predicts the side of the
> centerline better (ROC AUC about 0.62) than one on the gradient feature (about 0.54), and the odor
> sum has no predictive power.

> Fitted on all three features together in the complex plume, the logistic model gives motion a larger
> weight (about 0.40) than the gradient (about 0.10), and the sum a weight of 0; the combined model
> reaches a test AUC of 0.612.

Samples per the SI (Plume movies, Logistic regression models) and the code (centerline_models.py),
as the claims file gives them. Readings named in the claims file: the region (a) mean intensity above
5, (b) the Otsu mask (bias subtracted); the sampling (a) random windows split at random, (b) split by
location. A third sampling reading is added (author, step 1.6): each sample paired with its
left-right mirror, label flipped. The paper does not describe how samples are balanced, but it reports
a sum AUC of exactly 0.5 with SD 0 and a sum weight of 0 "as expected", which only a mirror-balanced
sample gives (the deposited plume is not symmetric about row 590). 10 repeats per reading (the paper
uses 100; 1000 new locations and 2,400,000 + 600,000 samples each, as the SI), seeds fixed.

In [ ]:
rng2 = np.random.default_rng(20260712)
N_REP = 10
regions = {"mean > 5": masks["mean > 5"], "Otsu": masks["Otsu, bias 48 subtracted"]}
loc = {k: [plumes.choose_locations(v, 1000, rng2) for _ in range(N_REP)] for k, v in regions.items()}
rows_all = np.concatenate([r for k in loc for r, c in loc[k]])
cols_all = np.concatenate([c for k in loc for r, c in loc[k]])
t0 = time.time()
traces = plumes.sample_antennas(movie, rows_all, cols_all)
print(f"pass 2: antennae at {len(rows_all)} locations over {traces.shape[0]} frames in {time.time() - t0:.0f} s;",
      "signal range", float(traces.min()), "to", float(traces.max()))

In [ ]:
t0 = time.time()
fig2, delays = [], []
for ri, region in enumerate(regions):
    for rep in range(N_REP):
        j = ri * N_REP + rep
        rr, cc = loc[region][rep]
        F = plumes.window_features(traces[:, j * 1000:(j + 1) * 1000], delays=range(1, 16))
        lab = (rr + 1.5 > plumes.COMPLEX_CENTERLINE).astype(int)
        for sampling in ("random", "location", "mirror"):
            Xtr, ytr, Xte, yte = plumes.draw_samples(F, lab, sampling, rng2)
            fig2.append({"region": region, "sampling": sampling, "repeat": rep, "right share": lab.mean(),
                         **plumes.fit_fig2(Xtr, ytr, Xte, yte)})
        if region == "mean > 5":    # the complex part of figs2c-one-frame-delay, shown below
            keys = tuple(f"motion{d}" for d in range(1, 16))
            Xtr, ytr, Xte, yte = plumes.draw_samples(F, lab, "mirror", rng2, keys=keys)
            for k, d in enumerate(range(1, 16)):
                delays.append({"repeat": rep, "delay": d, **plumes.fit_fig2(Xtr[:, [k]], ytr, Xte[:, [k]], yte, names=("motion",))})
        del F
F2 = pd.DataFrame(fig2)
print(f"{len(F2)} repeats fitted in {time.time() - t0:.0f} s; no fit failed (sklearn lbfgs, unpenalised)")
agg = F2.groupby(["region", "sampling"]).agg(["mean", "std"]).drop(columns=["repeat"])
agg.columns = [f"{a} {b}" for a, b in agg.columns]
print(agg.round(3).T.to_string())

In [ ]:
m = F2.groupby(["region", "sampling"]).mean(numeric_only=True)
rows = []
for (region, sampling), r in m.iterrows():
    rows.append({"region": region, "sampling": sampling,
                 "motion AUC": r.auc_motion, "gradient AUC": r.auc_gradient, "sum AUC": r.auc_sum,
                 "motion > gradient": r.auc_motion > r.auc_gradient,
                 "sum within 0.5 +/- 0.05": abs(r.auc_sum - 0.5) <= 0.05,
                 "motion within 0.618 +/- 0.005": abs(r.auc_motion - 0.618) <= 0.005,
                 "gradient within 0.537 +/- 0.012": abs(r.auc_gradient - 0.537) <= 0.012})
print("fig2c-complex-auc, means over repeats:")
print(pd.DataFrame(rows).round(3).to_string(index=False))
rows = []
for (region, sampling), r in m.iterrows():
    rows.append({"region": region, "sampling": sampling, "|w motion|": abs(r.w_motion), "|w gradient|": abs(r.w_gradient), "|w sum|": abs(r.w_sum),
                 "combined AUC": r.auc_all, "|motion| > |gradient|": abs(r.w_motion) > abs(r.w_gradient), "|sum| < 0.01": abs(r.w_sum) < 0.01,
                 "motion within 0.403 +/- 0.034": abs(abs(r.w_motion) - 0.403) <= 0.034,
                 "gradient within 0.098 +/- 0.041": abs(abs(r.w_gradient) - 0.098) <= 0.041,
                 "AUC within 0.612 +/- 0.061": abs(r.auc_all - 0.612) <= 0.061})
print("\nfig2e-complex-weights, means over repeats (weights of standardised features):")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nSD over repeats of the motion AUC:", F2.groupby(["region", "sampling"]).auc_motion.std().round(4).to_dict())

**Result (fig2c-complex-auc): partially reproduced, resting on an assumption.** Under every reading the
motion model predicts the side better than the gradient model (AUC 0.601 to 0.604 against 0.552 to
0.566). Under the mirror-balanced reading the sum has no predictive power (AUC 0.500, SD 0), as in the
paper; in the other two samplings the sum predicts the side (AUC 0.63 to 0.67), because the deposited
plume is not symmetric about row 590, and the criterion's sum condition fails. The motion AUC is below
0.618 +/- 0.005 and the gradient AUC above 0.537 +/- 0.012 under every reading, so the values do not
hold. The outcome rests on the mirror-balanced sampling.

**Result (fig2e-complex-weights): partially reproduced.** The |motion| weight (0.415 to 0.424) exceeds
the |gradient| weight (0.234 to 0.271) under every reading, and the sum weight is 0 under the
mirror-balanced reading (0.22 to 0.33 otherwise). The motion weight is within 0.403 +/- 0.034 under every
reading and the combined AUC within 0.612 +/- 0.061 under the mirror reading (0.611 and 0.614), but the
gradient weight is more than twice the expected 0.098 +/- 0.041 under every reading, so only the
ordering holds; every reading gives this outcome.

### Complex part of figs2c-one-frame-delay (blocked claim)

The claim needs both plumes; the smooth plume could not be read, so it is blocked. The complex part:
motion-only models with delays 1 to 15 frames, on the samples of each repeat (mirror-balanced, region
mean > 5).

In [ ]:
D2 = pd.DataFrame(delays).groupby("delay").auc_motion.agg(["mean", "std"])
print(D2.round(4).T.to_string())
sd1 = D2.loc[1, "std"]
print("no delay above delay 1 by more than its SD:", bool((D2["mean"].iloc[1:] <= D2.loc[1, "mean"] + sd1).all()),
      "; delay 15 below delay 1 by more than the SD:", bool(D2.loc[15, "mean"] < D2.loc[1, "mean"] - sd1),
      "; delay 1 the highest:", bool(D2["mean"].idxmax() == 1))

## Summary

| Claim | Outcome |
|---|---|
| fig5e-cue-betas | reproduced, assumed (SI onset, smoke-2a files); the complex motion coefficient has the opposite sign to the paper's, and the pattern fails with the files' turn start |
| fig5f-cue-dominance | reproduced, assumed (SI onset, smoke-2a files, code's complex heading window); positive complex dominance with the files' turn start |
| figs5c-turn-angles | reproduced, assumed (turn bounded by the 20% crossings); about -21 and +22 degrees over the files' events |
| fig2c-complex-auc | partially reproduced, assumed (mirror-balanced samples): ordering and sum hold, values do not |
| fig2e-complex-weights | partially reproduced: ordering holds, gradient weight too large |
| figs1fg-resolution-robustness | not reproduced: gradient r 0.61 at 12 px, motion r about 0.73 at 4 frames |
| methods-intensity-right-bias | blocked: metadata_insufficient |
| the 15 claims on the smooth plume, the networks and the agents | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The Fig. 5 analysis is the paper's code, ported (it gives
the code's numbers on the same files, checked in preparation) and run under the readings the claims file
names, with the SI's turn onset as the main reading. The Fig. 2 samples are drawn by a procedure the paper
does not give in full (mirror-balanced, assumed), with 10 repeats in place of 100. The motion cue of Fig.
S1G is not defined in the paper; two definitions are run.

**What was found about the dataset.** The complex-plume behavior files the code selects (smoke-2a),
run through the paper's own code, do not give the code's published complex Fig. 5E and 5F values (motion
-0.012 against +0.044; dominance +0.39 against -0.66); the smooth files give values close to the
published ones. The files do not say which complex files the paper used (the Dataverse deposit lists
both smoke-1a and smoke-2a as complex), which turn direction is left or right, or how the heading's
axes are oriented (theta is atan2(-vy, vx)). The turn events start a median of 4 frames (67 ms) after the SI's onset
and end before the turn does. Four smooth files have frames far outside the arena. The odor signals
cannot be aligned in time with the deposited movie. The smooth plume, needed by 15 claims, is in
another archive that requires a token to download.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")
import resource
print("peak memory", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")